In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import joblib

## 1. Data Loading and Preprocessing

In [7]:
df = pd.read_csv('/content/creditcard.csv')

# Assuming 'Class' is the target variable for credit card fraud datasets
# If your target column is named differently, please update 'Class' to your target column name.
if 'Class' in df.columns:
    df = df.rename(columns={'Class': 'target'})
else:
    print("Warning: 'Class' column not found, assuming 'target' already exists or will be created.")

for col in df.select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data loaded, preprocessed, and split into training and testing sets.")


Data loaded, preprocessed, and split into training and testing sets.


## 2. Model Definition

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", use_label_encoder=False, random_state=42)
}

print("Models defined.")


Models defined.


## 3. Model Training and Evaluation

In [9]:
best_model = None
best_acc = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name}: Accuracy = {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_model = model

print(f"\nTraining and evaluation complete. Best accuracy found: {best_acc:.4f}")


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: Accuracy = 0.9989
Decision Tree: Accuracy = 0.9991
Random Forest: Accuracy = 0.9996


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:48:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost: Accuracy = 0.9995

Training and evaluation complete. Best accuracy found: 0.9996


In [11]:
app_py_content = """
from flask import Flask, render_template, request
import joblib
import numpy as np

app = Flask(__name__)

model = joblib.load("credit_card_model.pkl")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    data = [float(x) for x in request.form.values()]
    pred = model.predict([data])[0]

    # Corrected logic: 1 usually indicates fraud, 0 indicates legitimate
    result = "Fraudulent Transaction" if pred == 1 else "Legitimate Transaction"

    return render_template("index.html", prediction=result)

if __name__ == "__main__":
    app.run(debug=True)
"""

with open("/content/app.py", "w") as f:
    f.write(app_py_content)

print("Updated /content/app.py with corrected prediction logic.")


Updated /content/app.py with corrected prediction logic.


## 4. Save Best Model

In [10]:
joblib.dump(best_model, "credit_card_model.pkl")
print(f"Best model saved to credit_card_model.pkl with accuracy: {best_acc:.4f}")


Best model saved to credit_card_model.pkl with accuracy: 0.9996
